# Chapter 1: Syntax and Semantics

This notebook follows Chapter 1 of [Boxes and Diamonds](https://bd.openlogicproject.org),
an open introduction to modal logic, using the **Gamen.jl** package.

We cover:
- The language of basic modal logic (Definition 1.1)
- Building formulas (Definition 1.2)
- Relational models (Definition 1.6)
- Truth at a world (Definition 1.7)
- Truth in a model (Definition 1.9)
- Validity and entailment

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using Gamen

## 1.1 The Language of Basic Modal Logic

The language of modal logic (Definition 1.1) contains:

1. The propositional constant for falsity: $\bot$
2. Propositional variables: $p_0, p_1, p_2, \ldots$
3. Propositional connectives: $\lnot, \land, \lor, \to$
4. The modal operator $\square$ (box / necessity)
5. The modal operator $\diamond$ (diamond / possibility)

### Atomic Formulas

We can create propositional variables by name or by index:

In [ ]:
# Named atoms
p = Atom(:p)
q = Atom(:q)
r = Atom(:r)

# Indexed atoms (Definition 1.1, item 2)
p0 = Atom(0)
p1 = Atom(1)

println("Named: ", p, ", ", q, ", ", r)
println("Indexed: ", p0, ", ", p1)

### Building Formulas (Definition 1.2)

Formulas are built inductively. Every atom is a formula, and if $A$ and $B$ are formulas, so are $\lnot A$, $(A \land B)$, $(A \lor B)$, $(A \to B)$, $\square A$, and $\diamond A$.

In [ ]:
# Falsity and truth (Definition 1.3)
println("⊥ = ", Bottom())
println("⊤ = ", Top(), "  (abbreviates ¬⊥)")

In [ ]:
# Propositional connectives
println("¬p       = ", Not(p))
println("p ∧ q    = ", And(p, q))
println("p ∨ q    = ", Or(p, q))
println("p → q    = ", Implies(p, q))
println("p ↔ q    = ", Iff(p, q), "  (abbreviates (p → q) ∧ (q → p))")

In [ ]:
# Modal operators
println("□p         = ", Box(p), "       (necessarily p)")
println("◇q         = ", Diamond(q), "       (possibly q)")
println()

# Nested formulas
println("□p → p     = ", Implies(Box(p), p))

# Schema K: □(A → B) → (□A → □B)
k_schema = Implies(Box(Implies(p, q)), Implies(Box(p), Box(q)))
println("Schema K   = ", k_schema)

### Modal-Free Formulas

A formula is *modal-free* if it contains no $\square$ or $\diamond$ operators:

In [ ]:
println("is_modal_free(p ∧ ¬q)   = ", is_modal_free(And(p, Not(q))))
println("is_modal_free(p → □q)  = ", is_modal_free(Implies(p, Box(q))))

## 1.4 Relational Models

A *model* $M = \langle W, R, V \rangle$ consists of (Definition 1.6):

1. $W$: a nonempty set of "worlds"
2. $R$: a binary accessibility relation on $W$
3. $V$: a valuation function assigning to each propositional variable $p$ the set $V(p) \subseteq W$ of worlds where $p$ is true

### Figure 1.1 from Boxes and Diamonds

The book's first example model has three worlds:

```
     w₂ (p, q)
    ↗
w₁ (p)
    ↘
     w₃ (¬p, ¬q)
```

- $W = \{w_1, w_2, w_3\}$
- $R = \{\langle w_1, w_2 \rangle, \langle w_1, w_3 \rangle\}$
- $V(p) = \{w_1, w_2\}$, $V(q) = \{w_2\}$

In [ ]:
frame = KripkeFrame([:w1, :w2, :w3], [:w1 => :w2, :w1 => :w3])
model = KripkeModel(frame, [:p => [:w1, :w2], :q => [:w2]])

## 1.5 Truth at a World (Definition 1.7)

The satisfaction relation $M, w \Vdash A$ ("$A$ is true at world $w$ in model $M$") is defined inductively:

| Clause | Rule |
|--------|------|
| 1 | $M, w \not\Vdash \bot$ (never) |
| 2 | $M, w \Vdash p$ iff $w \in V(p)$ |
| 3 | $M, w \Vdash \lnot B$ iff $M, w \not\Vdash B$ |
| 4 | $M, w \Vdash B \land C$ iff $M, w \Vdash B$ and $M, w \Vdash C$ |
| 5 | $M, w \Vdash B \lor C$ iff $M, w \Vdash B$ or $M, w \Vdash C$ |
| 6 | $M, w \Vdash B \to C$ iff $M, w \not\Vdash B$ or $M, w \Vdash C$ |
| 7 | $M, w \Vdash \square B$ iff $M, w' \Vdash B$ for **all** $w'$ with $Rww'$ |
| 8 | $M, w \Vdash \diamond B$ iff $M, w' \Vdash B$ for **some** $w'$ with $Rww'$ |

### Problem 1.1 — Which of the following hold?

Let's work through the book's exercises on Figure 1.1:

In [ ]:
problems = [
    "1. M,w₁ ⊩ q"         => satisfies(model, :w1, q),
    "2. M,w₃ ⊩ ¬q"        => satisfies(model, :w3, Not(q)),
    "3. M,w₁ ⊩ p ∨ q"     => satisfies(model, :w1, Or(p, q)),
    "4. M,w₁ ⊩ □(p ∨ q)"  => satisfies(model, :w1, Box(Or(p, q))),
    "5. M,w₃ ⊩ □q"        => satisfies(model, :w3, Box(q)),
    "6. M,w₃ ⊩ □⊥"        => satisfies(model, :w3, Box(Bottom())),
    "7. M,w₁ ⊩ ◇q"        => satisfies(model, :w1, Diamond(q)),
    "8. M,w₁ ⊩ □q"        => satisfies(model, :w1, Box(q)),
    "9. M,w₁ ⊩ ¬□□¬q"     => satisfies(model, :w1, Not(Box(Box(Not(q))))),
]

for (desc, result) in problems
    println(desc, "  =>  ", result)
end

**Understanding the results:**

- **Item 1** is false: $w_1 \notin V(q)$.
- **Item 4** is false: $w_3$ satisfies neither $p$ nor $q$, so $p \lor q$ fails there, and $\square(p \lor q)$ requires it at all accessible worlds.
- **Items 5 and 6** are *vacuously true*: $w_3$ has no accessible worlds, so $\square B$ holds for any $B$ at $w_3$.
- **Item 8** is false: $q$ is not true at $w_3$, which is accessible from $w_1$.
- **Item 9**: $\square\square\lnot q$ is true at $w_1$ because $w_2$ and $w_3$ have no successors, making $\square\lnot q$ vacuously true at both. So $\lnot\square\square\lnot q$ is false.

## Proposition 1.8 — Duality of □ and ◇

The book proves that $\square$ and $\diamond$ are duals:
- $M, w \Vdash \square A$ iff $M, w \Vdash \lnot\diamond\lnot A$
- $M, w \Vdash \diamond A$ iff $M, w \Vdash \lnot\square\lnot A$

Let's verify this holds at every world in our model:

In [ ]:
for w in [:w1, :w2, :w3]
    box_ok = satisfies(model, w, Box(p)) == satisfies(model, w, Not(Diamond(Not(p))))
    dia_ok = satisfies(model, w, Diamond(p)) == satisfies(model, w, Not(Box(Not(p))))
    println("$w:  □p ↔ ¬◇¬p = $box_ok,  ◇p ↔ ¬□¬p = $dia_ok")
end

## 1.6 Truth in a Model (Definition 1.9)

A formula $A$ is *true in a model* $M$ (written $M \Vdash A$) if it is true at every world in $M$:

In [ ]:
# ⊤ is true everywhere
println("M ⊩ ⊤    = ", is_true_in(model, Top()))

# p is not true in the model (false at w3)
println("M ⊩ p    = ", is_true_in(model, p))

# □⊥ is not true in the model (false at w1, which has successors)
println("M ⊩ □⊥   = ", is_true_in(model, Box(Bottom())))

## 1.9 Schemas and Validity

A *schema* is a set of formulas obtained by substituting into a characteristic formula (Definition 1.17). A schema is *valid* if it is true in every model.

### Proposition 1.19 — Schema K is valid

$\square(A \to B) \to (\square A \to \square B)$

We can't prove universal validity computationally, but we can verify it on specific models:

In [ ]:
# Schema K: □(p → q) → (□p → □q)
k_instance = Implies(Box(Implies(p, q)), Implies(Box(p), Box(q)))
println("Schema K instance: ", k_instance)
println("True in Figure 1.1 model: ", is_true_in(model, k_instance))

# Check against a few more models
frame2 = KripkeFrame([:a, :b, :c], [:a => :b, :b => :c, :a => :c])
model2 = KripkeModel(frame2, [:p => [:a, :b], :q => [:b, :c]])

frame3 = KripkeFrame([:w], [:w => :w])
model3 = KripkeModel(frame3, [:p => [:w], :q => [:w]])

println("Valid across test models: ", is_valid(k_instance, [model, model2, model3]))

### Table 1.1 — Valid and Invalid Schemas

The book lists several valid and invalid schemas. Let's check an invalid one:

$A \to \square A$ ("if $A$ is true, then $A$ is necessarily true")

This should fail — just because something is true at a world doesn't mean it's true at all accessible worlds:

In [ ]:
# A → □A is NOT valid
invalid_schema = Implies(p, Box(p))
println("A → □A: ", invalid_schema)
println("True in Figure 1.1 model: ", is_true_in(model, invalid_schema))

# Counterexample: at w1, p is true but □p is false (p is false at w3)
println()
println("Counterexample:")
println("  M,w₁ ⊩ p   = ", satisfies(model, :w1, p))
println("  M,w₁ ⊩ □p  = ", satisfies(model, :w1, Box(p)))
println("  M,w₃ ⊩ p   = ", satisfies(model, :w3, p), "  ← p fails at accessible world w₃")

## 1.10 Entailment (Definition 1.23)

A set of formulas $\Gamma$ *entails* $A$ in model $M$ if: for every world $w \in W$, if $M, w \Vdash B$ for every $B \in \Gamma$, then $M, w \Vdash A$.

In [ ]:
# Build a model where p and q hold everywhere
frame_e = KripkeFrame([:w1, :w2], [:w1 => :w2])
model_e = KripkeModel(frame_e, [:p => [:w1, :w2], :q => [:w1, :w2]])

# Single premise: p entails p ∨ q
println("p ⊨ p ∨ q       = ", entails(model_e, p, Or(p, q)))

# Multiple premises: {p, q} entails p ∧ q
println("{p, q} ⊨ p ∧ q  = ", entails(model_e, [p, q], And(p, q)))

## Exercises

Try these on your own:

1. **Build Figure 1.2** from the book: $W = \{w_1, w_2, w_3\}$, $R = \{\langle w_1, w_2 \rangle, \langle w_1, w_3 \rangle\}$, $V(p) = \{w_2, w_3\}$. Use it to show that $p \to \diamond p \vDash \square p \to p$ but $p \to \diamond p \not\vDash \square p \to p$ doesn't hold in the other direction.

2. **Valid schemas from Table 1.1**: Verify that $\square(A \to B) \to (\diamond A \to \diamond B)$ holds in several models.

3. **Proposition 1.10**: Show that if $M \Vdash A$ then $M \not\Vdash \lnot A$ (but not vice versa) and if $M \Vdash A \to B$ then $M \Vdash A$ only if $M \Vdash B$ (but not vice versa).

In [ ]:
# Scratch space for exercises